# Warped regression — §4.1 / fig. 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/02_toy_warped_regression.ipynb)

Port of Matlab [`demToyRegressionSimple.m`](https://github.com/SheffieldML/deepGP/blob/master/deepGP/matlab/demToyRegressionSimple.m) with
data from `hierGpsNEW` in `demToyDynamicsCreateData.m`.

Defaults: Ntoy=150, D=15, Ntr=25, H=2, Q=6, K=10 (Matlab simple). Set `PAPER = True`
for the AISTATS text sizes (N=120, D=10).


## Setup


In [ ]:
import sys, subprocess, os, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd().resolve()
# Prefer sibling of notebooks/ (repo layout)
for c in [ROOT.parent, ROOT, ROOT / "examples" / "damianou2013"]:
    if (c / "_common.py").is_file():
        ROOT = c
        break

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "matplotlib", "GPy>=1.15", "scikit-learn"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        f"git+https://github.com/SheffieldML/PyDeepGP.git@examples-damianou2013"])
    ROOT = Path.cwd().resolve()
    (ROOT / "data").mkdir(exist_ok=True)
    for dest, url in [
        ("_common.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_common.py"),
        ("_matlab_data.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_matlab_data.py"),
        ("data/hgplvmSampleDataTr1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/hgplvmSampleDataTr1.mat"),
        ("data/demHighFiveHgplvm1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/demHighFiveHgplvm1.mat"),
    ]:
        p = ROOT / dest
        if not p.is_file():
            print("Fetching", dest)
            urllib.request.urlretrieve(url, str(p))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import GPy
import deepgp
from _common import mse, optimize_staged, rbf_bias
from _matlab_data import generate_hier_gps_new, train_test_split_indices


In [ ]:
QUICK = True
PAPER = False  # True → N=120, D=10
n_toy, d_toy = (120, 10) if PAPER else (150, 15)
n_tr = 25
q = 3 if QUICK else 6
num_inducing = 8 if QUICK else 10
repeats = 2 if QUICK else 10
iters_fixed = 40 if QUICK else 500
iters_free = 120 if QUICK else 2000
SEED = 42
print(f"Ntoy={n_toy} D={d_toy} Ntr={n_tr} Q={q} K={num_inducing} repeats={repeats}")


## Generative process (`hierGpsNEW`)


In [ ]:
X, H, Y = generate_hier_gps_new(n_toy=n_toy, d_toy=d_toy, seed=SEED)
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
axes[0].plot(X[:, 0], H[:, 0], "k-")
axes[0].set_ylabel("hidden warp")
axes[0].set_title("hierGpsNEW generative process")
axes[1].plot(X[:, 0], Y[:, 0], label="Y[:,0]")
axes[1].plot(X[:, 0], Y[:, 1], label="Y[:,1]")
axes[1].set_xlabel("X (scaled)")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


## Fit sparse GP and DeepGP across repeats


In [ ]:
def fit_gp(Xtr, Ytr, num_inducing):
    kern = GPy.kern.RBF(Xtr.shape[1]) + GPy.kern.Bias(Xtr.shape[1])
    m = GPy.models.SparseGPRegression(Xtr, Ytr, kernel=kern, num_inducing=num_inducing)
    m.Gaussian_noise.variance = Ytr.var() * 0.01
    m.Gaussian_noise.variance.fix()
    m.optimize(max_iters=100, messages=False)
    m.Gaussian_noise.variance.unfix()
    m.optimize(max_iters=400, messages=False)
    return m

def fit_dgp(Xtr, Ytr, q, num_inducing, iters_fixed, iters_free):
    # H=2 hidden layers between observed X and Y
    dims = [Ytr.shape[1], q, q, Xtr.shape[1]]
    kernels = [rbf_bias(q, ard=True), rbf_bias(q, ard=True), rbf_bias(Xtr.shape[1], ard=False)]
    m = deepgp.DeepGP(
        dims, Ytr, X=Xtr, kernels=kernels,
        num_inducing=num_inducing, back_constraint=False,
    )
    optimize_staged(m, iters_fixed, iters_free, messages=False)
    return m


In [ ]:
gp_mses, dgp_mses = [], []
for r in range(repeats):
    seed = SEED + r
    X, H, Y = generate_hier_gps_new(n_toy=n_toy, d_toy=d_toy, seed=seed)
    ind_tr, ind_ts = train_test_split_indices(n_toy, n_tr, seed=seed + 1000)
    Xtr, Ytr = X[ind_tr], Y[ind_tr]
    Xte, Yte = X[ind_ts], Y[ind_ts]
    m_gp = fit_gp(Xtr, Ytr, num_inducing)
    m_dgp = fit_dgp(Xtr, Ytr, q, num_inducing, iters_fixed, iters_free)
    gp_mses.append(mse(m_gp.predict(Xte)[0], Yte))
    dgp_mses.append(mse(m_dgp.predict(Xte)[0], Yte))
    print(f"Repeat {r + 1}/{repeats}: MSE GP={gp_mses[-1]:.4f}  DeepGP={dgp_mses[-1]:.4f}")

print(
    f"Mean MSE (±std): GP {np.mean(gp_mses):.4f}±{np.std(gp_mses):.4f}  "
    f"DeepGP {np.mean(dgp_mses):.4f}±{np.std(dgp_mses):.4f}"
)


## Fig. 3(b)-style MSE plot


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
xs = np.arange(1, repeats + 1)
ax.plot(xs, gp_mses, "o-", label="sparse GP")
ax.plot(xs, dgp_mses, "s-", label="DeepGP")
ax.set_xlabel("experiment #")
ax.set_ylabel("test MSE")
ax.set_title("Fig. 3(b) style / demToyRegressionSimple")
ax.legend()
fig.tight_layout()
plt.show()
